# 🧱 Part 08: BERT Encoder: Bidirectional Understanding

> **Previous context**: GPT predicts the next token with a causal mask. BERT uses a different mask so tokens can read both directions.
> **Goal for this part**: Build a small encoder-only model and understand Masked Language Modeling (MLM).

Today we are solving one concrete confusion: what is the hidden mechanism behind this part of an LLM, and how can we rebuild it with small numbers before trusting a library?

## 0. Decoder vs encoder

GPT is good at continuation because it reads left to right. BERT is good at understanding because every token can attend to both left and right context.

## 1. MLM task

Instead of predicting the next token, BERT hides some tokens and asks the model to recover them from surrounding context.

## 2. Classification head

Encoder representations can feed tasks such as classification because they summarize the whole input with bidirectional context.

## 3. Key distinction

BERT is not a text generator in the same way GPT is; it is mainly an encoder for understanding tasks.

## How to use the code cells

Run the cells in order. The code is intentionally direct and small: each cell should expose one idea, print the key observation, and let you change a number to see what moves.

## Exercises

When a cell contains a TODO placeholder, fill it yourself and use the `assert` checks as feedback. You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

## Summary Checklist

- [ ] BERT uses bidirectional attention.
- [ ] MLM trains the model to recover masked tokens.
- [ ] Encoder outputs are useful for understanding and classification tasks.

Next, continue through the code cells for the Training Systems part and inspect the printed observations.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

### 1. Encoder vs Decoder: what is the *core* architectural difference?

You should have a picture like this in your head:

```
        Encoder-Only (BERT)          Decoder-Only (GPT)
        ----------------             ----------------

        Output: "B-PER"                 Output: "the"
          ^                               ^
    +-----+-----+                   +-----+-----+
    |  FFN + LN  |                  |  FFN + LN  |
    +-----------+                   +-----------+
    | Attention  |                  | Attention  |
    | (bi-dir!)  |                  | (causal!)  |
    +-----------+                   +-----------+
    |  Input     |                  |  Input     |
    +-----------+                   +-----------+

   each token can attend to          each token can only attend to
   all tokens (left and right)       previous tokens (causal mask)
```

Take the same word "apple":

- **What BERT sees**: "I ate an __ today." It can look left and right, so it knows this blank is a *noun* and likely a *food*.
- **What GPT sees**: "I ate an" It can only look left, and the training task is to guess the next token.

| | Encoder-Only (BERT) | Decoder-Only (GPT) | Encoder-Decoder (T5) |
|:---|:---|:---|:---|
| Attention direction | **bidirectional** (see the whole sentence) | **unidirectional** (only look left) | bi-dir encoder + causal decoder |
| Pretraining objective | **MLM** (masked language modeling) | **autoregressive** (next-token prediction) | span corruption |
| Best at | understanding, classification, tagging | generation, dialogue | translation, summarization |
| Representative models | BERT, RoBERTa, DeBERTa | GPT-3/4, LLaMA, Qwen | T5, BART |


### 2. BERT input representation: Token + Segment + Position

BERT's input is not a single embedding lookup. It is the **sum of three embeddings**:

```
Input = Token Embedding + Segment Embedding + Position Embedding
         ^                    ^                    ^
   "what the token is"   "sentence A or B"     "where it is"
```

How is this different from GPT? GPT typically uses Token + Position, and does not need Segment Embeddings (because it does not explicitly model "two-sentence relations" like BERT's original NSP setup).

**Why does BERT need Segment Embeddings?** Historically, because the NSP (Next Sentence Prediction) task needs the model to know which tokens came from sentence A vs sentence B.

Next, we'll build the full BERT input step by step in code.


In [ ]:
# ============================================================
# Manually construct BERT inputs (from scratch, step by step)
# ============================================================

# A tiny BERT configuration for demonstration
VOCAB_SIZE = 100
D_MODEL = 16
MAX_LEN = 20
NUM_SEGMENTS = 2  # sentence A vs sentence B

# Three embeddings
token_embed = nn.Embedding(VOCAB_SIZE, D_MODEL)
segment_embed = nn.Embedding(NUM_SEGMENTS, D_MODEL)
position_embed = nn.Embedding(MAX_LEN, D_MODEL)

# Simulated input: two sentences
#   [CLS] I like cats [SEP] he likes dogs [SEP]
#   where [CLS]=1 and [SEP]=2

sentence_A = [1, 5, 8, 3, 2]    # [CLS] I like cats [SEP]
sentence_B = [6, 8, 4, 2]        # he likes dogs [SEP]
full_ids = sentence_A + sentence_B
seq_len = len(full_ids)

print("Step 1: Token IDs")
print(f"  Input: {full_ids}")
print(f"  Meaning: [CLS] I like cats [SEP] he likes dogs [SEP]")

# Segment IDs: sentence A tokens are 0, sentence B tokens are 1
segment_ids = [0] * len(sentence_A) + [1] * len(sentence_B)
print(f"\nStep 2: Segment IDs")
print(f"  Segments: {segment_ids}")
print(f"  Meaning: first {len(sentence_A)} tokens are sentence A (0), last {len(sentence_B)} tokens are sentence B (1)")

# Position IDs：0, 1, 2, ..., seq_len-1
position_ids = list(range(seq_len))
print(f"\nStep 3: Position IDs")
print(f"  Positions: {position_ids}")

# ============================================================
# Sum the three embeddings
# ============================================================
tokens_t = torch.tensor(full_ids)
segments_t = torch.tensor(segment_ids)
positions_t = torch.tensor(position_ids)

tok_emb = token_embed(tokens_t)    # (seq_len, D_MODEL)
seg_emb = segment_embed(segments_t)  # (seq_len, D_MODEL)
pos_emb = position_embed(positions_t)  # (seq_len, D_MODEL)

input_embeddings = tok_emb + seg_emb + pos_emb

print(f"\nStep 4: sum the three embeddings")
print(f"  Token Embedding shape:    {tok_emb.shape}")
print(f"  Segment Embedding shape:  {seg_emb.shape}")
print(f"  Position Embedding shape: {pos_emb.shape}")
print(f"  After summation: {input_embeddings.shape}")

# Show: the same token (id=8) ends up with different embeddings in different segment/position
# token id=8 is at position 2 in sentence A, and position 1 in sentence B
idx_a = 2  # token id=8 in sentence A
idx_b = len(sentence_A) + 1  # token id=8 in sentence B

print(f"\nKey proof: the same token id differs with position + segment")
print(f"  token id=8 in sentence A (pos={idx_a}, seg=0) final embedding (first 6 dims):")
print(f"    {input_embeddings[idx_a, :6].detach()}")
print(f"  token id=8 in sentence B (pos={idx_b}, seg=1) final embedding (first 6 dims):")
print(f"    {input_embeddings[idx_b, :6].detach()}")
print(f"  Are they identical? {(input_embeddings[idx_a] == input_embeddings[idx_b]).all().item()}")
print(f"  -> Even if the token id is the same, position and segment make the final embedding different.")

### 3. The core of BERT: MLM (Masked Language Modeling)

GPT is trained with an autoregressive objective: "given the previous tokens, predict the next token".

BERT is trained with MLM: "mask out some tokens in the middle and ask the model to guess them".

```
GPT training:   I -> love -> you -> China
                can only look ->

BERT training:  I [MASK] you China  ->  guess [MASK] = "love"
                look left and right
```

BERT can look both left and right because its Attention is **bidirectional** (no causal mask).
That makes it strong at *understanding*, but it is not naturally a *generator* because it was never trained to produce tokens one by one.


In [ ]:
# ============================================================
# Demo: bidirectional attention (BERT) vs causal attention (GPT)
# ============================================================
seq_len = 6

# GPT causal mask: each position can only see itself and previous tokens
causal_mask = torch.tril(torch.ones(seq_len, seq_len))

# BERT has no causal mask: each position can see all tokens (bidirectional)
bert_mask = torch.ones(seq_len, seq_len)

tokens = ["[CLS]", "I", "love", "you", "China", "[SEP]"]

print("=== GPT causal attention (one-way) ===")
print(f"Tokens: {tokens}")
print()
for i in range(seq_len):
    visible = [tokens[j] for j in range(seq_len) if causal_mask[i, j] == 1]
    print(f"  pos {i} ('{tokens[i]}') can see: {visible}")

print(f"\n=== BERT attention (bidirectional) ===")
for i in range(seq_len):
    visible = [tokens[j] for j in range(seq_len) if bert_mask[i, j] == 1]
    print(f"  pos {i} ('{tokens[i]}') can see: {visible}")

print(f"\nKey difference:")
print(f"  GPT:  'you' cannot see 'China' (not generated yet)")
print(f"  BERT: 'you' can see 'China' (sentence is fully known)")
print(f"  -> BERT is stronger at understanding context, but it is not trained to generate.")

### 4. End-to-end demo: how MLM training works

We'll build a MiniBERT and run a complete MLM training flow:

1. Randomly mask 15% of tokens
2. Predict the masked tokens
3. Compute the loss only on the masked positions


In [ ]:
# ============================================================
# MiniBERT: encoder-only (bidirectional attention, no causal mask)
# ============================================================
class MiniBERTEncoder(nn.Module):
    """A Transformer block like MiniGPT, but without a causal mask (bidirectional)."""
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, S, D = x.shape
        Q = self.W_Q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        # Key: no causal mask (bidirectional)
        attn = F.softmax(scores, dim=-1)
        out = (attn @ V).transpose(1, 2).contiguous().view(B, S, D)
        return self.W_O(out)

class MiniBERTBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attention = MiniBERTEncoder(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class MiniBERT(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=2, max_len=64):
        super().__init__()
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        # BERT uses learned positional embeddings (not sinusoidal)
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([
            MiniBERTBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        self.norm_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)
        self.max_len = max_len

    def encode(self, x):
        """Return hidden states at every position for reuse in classification/QA tasks."""
        B, S = x.shape
        positions = torch.arange(S, device=x.device).unsqueeze(0)
        x_emb = self.token_embedding(x) + self.position_embedding(positions)
        for block in self.blocks:
            x_emb = block(x_emb)
        x_emb = self.norm_final(x_emb)
        return x_emb

    def forward(self, x):
        x_emb = self.encode(x)
        return self.lm_head(x_emb)

print("MiniBERT defined (bidirectional attention, no causal mask)")

In [ ]:
# ============================================================
# Build MLM training data + a tiny training demo
# ============================================================
VOCAB_SIZE = 50
MASK_ID = 3  # [MASK] token id

torch.manual_seed(42)
model = MiniBERT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Toy training sentences
sentences = [
    [5, 8, 10, 6, 9, 4],   # I like cats, they are very cute
    [5, 12, 13, 14, 4],    # I hate rainy days
    [7, 8, 11, 6, 15, 4],  # He likes dogs, they are loyal
    [5, 16, 17, 18, 4],    # I am learning programming
    [7, 16, 19, 20, 4],    # He is listening to music
    [5, 21, 22, 9, 23, 4], # I think math is hard
]

# Create MLM batches
def create_mlm_batch(sentences, mask_prob=0.15):
    """
    Same idea as real BERT:
    - 15% of tokens are selected
    - 80% replaced with [MASK]
    - 10% replaced with a random token
    - 10% kept unchanged (but still predicted)
    """
    # Padding
    max_len = max(len(s) for s in sentences)
    batch_size = len(sentences)
    input_ids = torch.zeros(batch_size, max_len, dtype=torch.long)
    labels = torch.full((batch_size, max_len), -100, dtype=torch.long)  # -100 = ignore

    for i, sent in enumerate(sentences):
        input_ids[i, :len(sent)] = torch.tensor(sent)

        # Select masked positions (15%), but do not mask the first/last token
        maskable = list(range(1, len(sent) - 1))
        num_mask = max(1, int(len(maskable) * mask_prob))
        mask_positions = torch.randperm(len(maskable))[:num_mask]

        for pos_idx in mask_positions:
            pos = maskable[pos_idx]
            labels[i, pos] = sent[pos]  # record the original token
            rand = torch.rand(1).item()
            if rand < 0.8:
                input_ids[i, pos] = MASK_ID  # 80%: replace with [MASK]
            elif rand < 0.9:
                input_ids[i, pos] = torch.randint(5, 25, (1,)).item()  # 10%: random token
            # 10%: keep unchanged

    return input_ids, labels

# Create a batch
input_ids, labels = create_mlm_batch(sentences)

print("=== MLM training batch example ===")
print(f"MASK_ID = {MASK_ID}")
print()
for i in range(len(sentences)):
    print(f"Sentence {i+1}:")
    print(f"  Original: {sentences[i]}")
    print(f"  Input:    {input_ids[i].tolist()}")
    label_row = labels[i].tolist()
    print(f"  labels: {[('MASK->'+str(l) if l != -100 else 'ignore') for l in label_row]}")
    print()

# ============================================================
# Train
# ============================================================
print("=== Training MiniBERT (MLM) ===")
NUM_EPOCHS = 200
losses = []

model.train()
for epoch in range(NUM_EPOCHS):
    optimizer.zero_grad()
    logits = model(input_ids)  # (B, S, V)
    loss = F.cross_entropy(
        logits.view(-1, VOCAB_SIZE),
        labels.view(-1),
        ignore_index=-100  # ignore non-masked positions
    )
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if epoch % 20 == 0 or epoch == NUM_EPOCHS - 1:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")

print(f"\nInitial loss: {losses[0]:.4f} -> final loss: {losses[-1]:.4f}")

In [ ]:
# ============================================================
# Test: MLM prediction
# ============================================================
test_sentence = torch.tensor([[5, MASK_ID, 10, 6, MASK_ID, 4]])  # I [MASK] cat it [MASK] cute

model.eval()
with torch.no_grad():
    logits = model(test_sentence)
    predictions = logits.argmax(dim=-1)

print("=== MLM prediction ===")
print(f"Input:  {test_sentence.tolist()[0]}")
print(f"       ['I', '[MASK]', 'cat', 'it', '[MASK]', 'cute']")
print(f"Pred:   {predictions.tolist()[0]}")
print(f"       ['I', '?', 'cat', 'it', '?', 'cute']")
print(f"\npos 1 (pred) = {predictions[0, 1].item()} (expected 8='like')")
print(f"pos 4 (pred) = {predictions[0, 4].item()} (expected 9='very')")

# Visualize loss
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3))
plt.plot(losses, 'b-', linewidth=1)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('MiniBERT MLM Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5. Fine-tuning BERT: how do downstream tasks attach to it?

After pretraining, you typically keep the BERT backbone and add a small task head on top.

| Task | How to use BERT | Where the head connects |
|:---|:---|:---|
| **single-sentence classification** (sentiment) | feed one sentence, use the `[CLS]` output | `[CLS]` hidden -> classifier |
| **sentence-pair classification** (NLI, similarity) | concatenate two sentences with `[SEP]` | `[CLS]` hidden -> classifier |
| **sequence tagging** (NER) | feed one sentence | each token hidden -> token-wise classifier |
| **question answering** (SQuAD) | question + passage | each token hidden -> two heads (start/end) |

Key insight: **the backbone stays the same (or is lightly fine-tuned); you mainly swap the head.**
This is a different mindset from GPT-style prompt engineering.


In [ ]:
# ============================================================
# Demo: sentence classification with MiniBERT
# ============================================================
class MiniBERTForClassification(nn.Module):
    """Attach a classification head on top of MiniBERT."""
    def __init__(self, bert, num_classes):
        super().__init__()
        self.bert = bert
        self.classifier = nn.Linear(bert.d_model, num_classes)

    def forward(self, x):
        hidden = self.bert.encode(x)  # (B, S, D)
        cls_output = hidden[:, 0, :]  # take the [CLS] output
        return self.classifier(cls_output)  # -> (B, num_classes)

# Demo
VOCAB_SIZE = 50
bert_base = MiniBERT(VOCAB_SIZE, d_model=64, num_heads=4, num_layers=2)
clf_model = MiniBERTForClassification(bert_base, num_classes=2)  # binary classification: positive/negative

test_input = torch.randint(0, VOCAB_SIZE, (1, 10))
output = clf_model(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {output.shape}  <- (batch=1, classes=2)")
print(f"Output logits: {output.tolist()}")
print(f"\nThis is the full BERT sentiment classification flow:")
print(f"  input -> BERT encoder -> [CLS] -> classifier -> label")

### 6. Loading a real BERT (via `transformers`)

We implemented a MiniBERT above. Now let's see how people use a real BERT model in practice.


In [ ]:
# ============================================================
# Load a real BERT via transformers
# ============================================================
try:
    from transformers import AutoTokenizer, AutoModel

    print("Loading bert-base-chinese...")
    tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")
    model = AutoModel.from_pretrained("bert-base-chinese")

    print(f"BERT parameter count: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")
    print(f"Vocab size: {len(tokenizer)}")
    print()

    # Test: inspect BERT attention weights
    sentences = [
        "I love China",
        "The weather is great today",
    ]

    inputs = tokenizer(sentences, padding=True, return_tensors="pt")
    print(f"Input IDs shape: {inputs['input_ids'].shape}")
    print(f"Attention mask: {inputs['attention_mask']}")
    print()

    # Forward pass
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    print(f"Output last_hidden_state shape: {outputs.last_hidden_state.shape}")
    print(f"  -> (batch, seq_len, hidden_dim=768)")
    print()

    # Look at the last layer attention (head 0)
    last_layer_attn = outputs.attentions[-1]  # (batch, num_heads, seq_len, seq_len)
    print(f"Last-layer attention shape: {last_layer_attn.shape}")
    print(f"  -> (batch, 12 heads, seq_len, seq_len)")
    print()

    # Show which tokens each token attends to (sentence 1)
    tokens1 = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    print(f"Sentence 1 tokens: {tokens1}")
    print(f"\nPer-token attention distribution (head 0):")
    for i, tok in enumerate(tokens1):
        attn_weights = last_layer_attn[0, 0, i]  # sample 0, head 0, position i
        top2 = attn_weights.topk(2)
        print(f"  '{tok}' attends most to: ", end="")
        for j, (idx, w) in enumerate(zip(top2.indices, top2.values)):
            print(f"'{tokens1[idx]}'({w:.2f})", end="  ")
        print()

except ImportError:
    print("transformers is not installed. Run: pip install transformers")
except Exception as e:
    print(f"Error while loading BERT: {e}")
    print("(This is expected if the network is unavailable or the model is too large; rely on the MiniBERT demo above.)")

### 7. BERT vs GPT: the essential differences in one table

| | BERT (Encoder-Only) | GPT (Decoder-Only) |
|:---|:---|:---|
| **core goal** | understanding: "what does this mean?" | generation: "what comes next?" |
| **pretraining** | MLM (mask 15% tokens) | autoregressive (next-token) |
| **attention** | bidirectional (see the whole sentence) | causal/unidirectional (only look left) |
| **input representation** | Token + Segment + Position | Token + Position (no Segment) |
| **output** | hidden states at every position | logits at every position (but generation uses the tail) |
| **how it's used** | add a task head and fine-tune | change prompts / chat formatting |
| **representatives** | BERT, RoBERTa, DeBERTa | GPT-3/4, LLaMA, Qwen, DeepSeek |
| **why GPT won** | BERT does not generate naturally; scaling did not yield the same emergent capabilities | GPT can do understanding via in-context learning |

> The ideas did not disappear. The mask-and-predict strategy shows up in later models (T5, BART) and even in multimodal setups.
> - RoBERTa removed NSP, kept MLM, trained longer on more data -> strong gains
> - DistilBERT distilled BERT to be smaller and faster
> - DeBERTa improved the attention mechanism (disentangled attention) and pushed benchmarks

**One-sentence summary**: BERT and GPT are two ways to use the same Transformer paper.
BERT uses the Encoder for understanding; GPT uses the Decoder for generation.
GPT won the scaling race, but bidirectional attention and MLM-style pretraining still matter across NLP.
